**Import libraries**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

**read Artifact**

In [ ]:
ml_table = pd.read_parquet('ml_orders_labeled.parquet')

**Monthly Late Rate Trend**

In [ ]:
# Work on a copy so the original ml_table is not modified
analysis_table = ml_table.copy()
analysis_table['year_month'] = analysis_table['order_purchase_timestamp'].dt.to_period('M')

# Calculate monthly late order rate
monthly_trend = analysis_table.groupby('year_month')['is_late'].agg(
    total_orders='count',
    late_orders='sum',
    late_rate=lambda x: x.mean() * 100
).reset_index()

monthly_trend['year_month_str'] = monthly_trend['year_month'].astype(str)

plt.figure(figsize=(12, 5))
plt.plot(monthly_trend['year_month_str'], monthly_trend['late_rate'], marker='o', color='#e74c3c', linewidth=2)

plt.title('Monthly Late Order Rate (%) Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Purchase Month', fontsize=11)
plt.ylabel('Late Orders Percentage (%)', fontsize=11)
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()

From the chart, we conclude that time-based splitting is not suitable due to the frequent spikes and instability in the late delivery rate. Therefore, we preferred using the Stratified Random Split method.

**Stratified Random Split**

In [ ]:
train_df, temp_df = train_test_split(
    ml_table, test_size=0.30, random_state=42, stratify=ml_table['is_late']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['is_late']
)

**date range and label balance**

In [ ]:
# Check date range and label balance for each split
date_column = 'order_purchase_timestamp'
label_column = 'is_late'

splits = {
    'train': train_df,
    'validation': val_df,
    'test': test_df
}

for split_name, split_df in splits.items():
    date_values = pd.to_datetime(split_df[date_column], errors='coerce')
    label_counts = split_df[label_column].value_counts().sort_index()
    label_percentages = split_df[label_column].value_counts(normalize=True).sort_index() * 100

    print(f'\n--- {split_name.capitalize()} split ---')
    print(f'Rows: {len(split_df)}')
    print(f'Date range: {date_values.min()} to {date_values.max()}')
    print('Label balance:')
    for label, count in label_counts.items():
        print(f'  {label}: {count} ({label_percentages[label]:.2f}%)')

**Artifact:ML splited tables**

In [ ]:
train_df.to_parquet('train.parquet', index=False)
val_df.to_parquet('val.parquet', index=False)
test_df.to_parquet('test.parquet', index=False)